**Special/Dunder Methods in Python**

##### `__getitem__` & `__len__`

In [ ]:
# -- __getitem__ & __len__ --

import collections
from random import choice

Book = collections.namedtuple("Book", ["title", "author", "year"])

class myBookShelf:
    def __init__(self, books):
        self._books = books
    def __getitem__(self, index):
        return self._books[index]
    def __len__(self):
        return len(self._books)

bookshelf1 = myBookShelf([
    Book("The Great Gatsby", "F. Scott Fitzgerald", 1925),
    Book("To Kill a Mockingbird", "Harper Lee", 1960),
    Book("1984", "George Orwell", 1949),
])

# because of __len__ 
# we can use len() function to get the number of books in the bookshelf
print(len(bookshelf1)) # Output: 3
print("------")

# because of __getitem__, 
# we can use indexing, slicing & iteration to access the books in the bookshelf easily 
print(bookshelf1[0]) 
print(bookshelf1[:2])
print(choice(bookshelf1))
print("------")

for book in bookshelf1:
    print(book)
print("------")
for book in reversed(bookshelf1):
    print(book)

3
------
Book(title='The Great Gatsby', author='F. Scott Fitzgerald', year=1925)
[Book(title='The Great Gatsby', author='F. Scott Fitzgerald', year=1925), Book(title='To Kill a Mockingbird', author='Harper Lee', year=1960)]
Book(title='1984', author='George Orwell', year=1949)
------
Book(title='The Great Gatsby', author='F. Scott Fitzgerald', year=1925)
Book(title='To Kill a Mockingbird', author='Harper Lee', year=1960)
Book(title='1984', author='George Orwell', year=1949)
------
Book(title='1984', author='George Orwell', year=1949)
Book(title='To Kill a Mockingbird', author='Harper Lee', year=1960)
Book(title='The Great Gatsby', author='F. Scott Fitzgerald', year=1925)


Using special methods like `__getitem__`, `__len__` allows us to use standard python methods for various operations as shown above. 

It helps us easily enable those pythonic standard methods to be used on our custom classes. And users of our class don't have to memorize arbitrary method names that we define.

This means we can use composition of Pydantic model and our domain class to create powerful domain objects. By separation of concern, Pydantic model will handle the correctness of data, and domain class handle how we work with it. See example below:

In [ ]:
from pydantic import BaseModel

class UserData(BaseModel):
    name: str
    age: int
    email: str
    scores: list[int]

class Users:
    def __init__(self, users):
        self._users = users
    def __getitem__(self, index):
        return self._users[index]
    def __len__(self):
        return len(self._users)

alice_data = UserData(
    name="Alice",
    age=30,
    email="alice@example.com",
    scores=[85, 90, 78]
)
bob_data = UserData(
    name="Bob",
    age="25",
    email="bob@example.com",
    scores=[92, 88, 95]
)

my_users = Users([alice_data, bob_data])
print(len(my_users))
print(my_users[0])

# attribute access works because UserData is a Pydantic model, 
# which allows attribute access to its fields
print(my_users[1].name) 

# however this will not work because 
#  the __getitem__ method is not defined for UserData class
print(my_users[0]["name"]) 

2
name='Alice' age=30 email='alice@example.com' scores=[85, 90, 78]
Bob


TypeError: 'UserData' object is not subscriptable

In [ ]:
# to fix above error, we can define __getitem__ method in UserData class
class UserData(BaseModel):
    name: str
    age: int
    email: str
    scores: list[int]
    def __getitem__(self, key):
        return getattr(self, key)

alice_data = UserData(
    name="Alice",
    age=30,
    email="alice@example.com",
    scores=[85, 90, 78]
)
bob_data = UserData(
    name="Bob",
    age="25",
    email="bob@example.com",
    scores=[92, 88, 95]
)

my_users = Users([alice_data, bob_data])
print(my_users[0]["name"]) # Output: Alice

Alice


##### `__iter__` & `__getitem__`

Both can make an object iterable, but they do it in different ways:

- `__iter__` → returns an iterator object <br>
- `__getitem__` → returns items by index until IndexError

Modern usage prefers `__iter__`. If it is missing, Python falls back to `__getitem__`.

In [ ]:
class myNumbers:
	def __init__(self, numbers):
		self.numbers = numbers
	def __len__(self):
		return len(self.numbers)
	def __iter__(self):
		print("using __iter__")
		return iter(self.numbers)

numbers = myNumbers([1, 2, 3, 4, 5])

# iter enables iteration over the numbers in the myNumbers class
for num in numbers:
    print(num)

# but it does not enable indexing, so the following will raise an error
print(numbers[2])

using __iter__
1
2
3
4
5


TypeError: 'myNumbers' object is not subscriptable

In [ ]:
class myNumbers:
    def __init__(self, numbers):
        self.numbers = numbers
    def __len__(self):
        return len(self.numbers)
    def __getitem__(self, index):
        print("using __getitem__")
        return self.numbers[index]
numbers = myNumbers([1, 2, 3, 4, 5])

# getitem enables both iteration and indexing, so the following will work without error
# however, compared to __iter__, __getitem__ is less efficient for iteration 
# because it creates a new iterator object for each iteration,
for num in numbers:
    print(num)
print(numbers[2])

using __getitem__
1
using __getitem__
2
using __getitem__
3
using __getitem__
4
using __getitem__
5
using __getitem__
using __getitem__
3


In [ ]:
# if both __iter__ and __getitem__ are defined,
# the __iter__ method will be used for iteration, 
# and the __getitem__ method will be used for indexing
class myNumbers:
    def __init__(self, numbers):
        self.numbers = numbers
    def __len__(self):
        return len(self.numbers)
    def __iter__(self):
        print("using __iter__")
        return iter(self.numbers)
    def __getitem__(self, index):
        print("using __getitem__")
        return self.numbers[index]
numbers = myNumbers([1, 2, 3, 4, 5])
for num in numbers:
    print(num)
print(numbers[2])

using __iter__
1
2
3
4
5
using __getitem__
3


##### Special Methods for Numeric Classes

- `__repr__`
- `__abs__`, `__add__`, `__mul__` 
- `__bool__`

In [ ]:
from collections import namedtuple

Vector = namedtuple("Vector", ["x", "y"])
v1 = Vector(1, 2)
v2 = Vector(3, 4)

# below behaviour is wrong because it does not perform vector addition as expected,
# but rather concatenates the two namedtuples into a new one with fields x, y, x, y.
print(v1 + v2) 

v1.add(v2) # This would raise an AttributeError since namedtuple instances are immutable

(1, 2, 3, 4)


AttributeError: 'Vector' object has no attribute 'add'

In [ ]:
# create our vector class allowing mathematical operations
from pydantic import BaseModel
import math

class Coordinate(BaseModel):
	x: float
	y: float


class Vector:
	def __init__(self, coord: Coordinate):
		self.coord = coord
  
	@property
	def x(self):
		# this allows us to access the x coordinate of the vector
		# using v.x instead of v.coord.x
		return self.coord.x
	@property
	def y(self):
		# likewise for y coordinate
		return self.coord.y

	def __repr__(self):
		return f"Vector({self.x!r}, {self.y!r})"

	def __abs__(self):
		# calculates the magnitude of a vector
		return math.hypot(self.x, self.y)

	def __bool__(self):
		# gives boolean state representation to the class
		return bool(abs(self))

	def __add__(self, other: "Vector"):
		# allows vector addition
		x = self.x + other.x
		y = self.y + other.y
		return Vector(Coordinate(x=x, y=y))

	def __mul__(self, scalar):
		# enables scalar multiplication
		x = self.x * scalar
		y = self.y * scalar
		return Vector(Coordinate(x=x, y=y))

In [ ]:
v1 = Vector(Coordinate(x=1, y=2))
v2 = Vector(Coordinate(x=3, y=4))
print(v1 + v2) # Output: Vector(4, 6)
print(v1 * 2) # Output: Vector(2, 4)
print(v1) # Output: Vector(1, 2)
print(abs(v1)) # Output: 2.23606797749979
print(bool(v1)) # Output: True

Vector(4.0, 6.0)
Vector(2.0, 4.0)
Vector(1.0, 2.0)
2.23606797749979
True


#####

**ref**: [Fluent Python (2nd Ed), Luciano Ramalho — Chapter 1](https://elmoukrie.com/wp-content/uploads/2022/05/luciano-ramalho-fluent-python_-clear-concise-and-effective-programming-oreilly-media-2022.pdf)

!["screenshot1 from Fluent Python Chapter 1"](assets/ch_01_image_1.png)

!["screenshot2 from Fluent Python Chapter 1"](assets/ch_01_image_2.png)